<a href="https://colab.research.google.com/github/Sneha-Tripathi/infosys-maritime-ai/blob/Milestone1/Milestone1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q streamlit streamlit-option-menu bcrypt pyngrok PyJWT


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 40.5 MB/s eta 0:00:00


In [9]:

%%writefile app.py
import os
import re
import sqlite3
import secrets
import smtplib
import datetime as dt
from email.message import EmailMessage
import bcrypt
import jwt
import streamlit as st
from streamlit_option_menu import option_menu
DB_PATH = "infosys_portal.db"
JWT_SECRET = os.getenv("JWT_SECRET", "")
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS", "")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD", "")
OTP_EXPIRY_SECONDS = 300
ADMIN_EMAIL = os.getenv("ADMIN_EMAIL", "").strip().lower()
SECURITY_QUESTIONS = ["What is your pet name?","Who is your Best Friend","What is your mother's maiden name?","What is your favourite city?",]

st.set_page_config(page_title="FreightQuote AI",page_icon="🚢",layout="wide",initial_sidebar_state="expanded",)
st.markdown(
    """
    <style>
    @import url(
        'https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap'
    );

    :root {--background: #F4F9FF;--surface: #FFFFFF;--primary: #3478F6;--primary-dark: #2458B8;--yellow: #FFD966;
        --text: #263858;--muted: #70809A;--border: #DCE8F5;--soft-blue: #E8F3FF;--soft-yellow: #FFF8DB;}
    .stApp {background: var(--background);color: var(--text);font-family: 'Inter', sans-serif;}
    [data-testid="stAppViewContainer"] {background: var(--background);}
    [data-testid="stHeader"] {background: rgba(244, 249, 255, 0.95);}
    .main .block-container {max-width: 1200px;padding-top: 2rem;padding-bottom: 3rem;}
    [data-testid="stSidebar"] {
        background: linear-gradient(180deg,#E7F3FF 0%,#F2F9FF 60%,#FFFBEA 100%);
        border-right: 1px solid var(--border);
    }
    [data-testid="stSidebar"] * {color: var(--text);}
    h1, h2, h3 {color: #203B69 !important;font-family: 'Inter', sans-serif !important;font-weight: 800 !important;}
    p, label, span {font-family: 'Inter', sans-serif;}
    .subtitle {color: var(--muted);font-size: 0.98rem; line-height: 1.7;}
    .brand-name {color: #2458B8;font-size: 2.2rem;font-weight: 800;letter-spacing: -1px;margin-bottom: 0.3rem;}
    .brand-tagline {color: var(--muted);font-size: 0.95rem;margin-bottom: 1.5rem;}
    .custom-card {background: var(--surface);border: 1px solid var(--border);border-radius: 18px;
        padding: 24px;box-shadow: 0 8px 28px rgba(48, 91, 145, 0.06);margin-bottom: 18px;}
    .auth-card {background: #FFFFFF;border: 1px solid var(--border);border-radius: 22px;padding: 30px;
        box-shadow: 0 12px 35px rgba(48, 91, 145, 0.08);}
    .metric-card {background: #FFFFFF;border: 1px solid var(--border);border-radius: 16px;
        padding: 20px;min-height: 115px;box-shadow: 0 6px 20px rgba(48, 91, 145, 0.05);}
    .metric-label {color: var(--muted);font-size: 0.9rem;margin-bottom: 8px;}
    .metric-value {color: #2458B8;font-size: 1.7rem;font-weight: 800;}
    .stButton > button {border-radius: 10px;min-height: 42px;font-weight: 600;border: 1px solid #C8DDF5;transition: all 0.2s ease-in-out;}
    .stButton > button[kind="primary"] {background: var(--primary);color: #FFFFFF;border: 1px solid var(--primary);}
    .stButton > button[kind="primary"]:hover {background: var(--primary-dark);border-color: var(--primary-dark);color: #FFFFFF;}
    .stButton > button:hover {border-color: var(--primary);color: var(--primary-dark);}
    .stTextInput input,
    .stPasswordInput input,
    .stSelectbox div[data-baseweb="select"] {border-radius: 10px;}
    [data-baseweb="input"] {border-radius: 10px;border-color: var(--border);}
    [data-testid="stAlert"] {border-radius: 12px;}
    hr {border-color: var(--border);}
    a {color: var(--primary-dark);}
    [data-testid="stProgress"] > div > div {background-color: var(--primary);}
    [data-testid="stDataFrame"] {border: 1px solid var(--border);border-radius: 12px;overflow: hidden;}
    footer {visibility: hidden;}
    </style>
    """,
    unsafe_allow_html=True,
)
def get_connection():
    """Create and return a SQLite database connection."""
    connection = sqlite3.connect(DB_PATH,check_same_thread=False,)
    connection.row_factory = sqlite3.Row
    return connection
def init_db():
    """Create the users table and update older database schemas."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS users (id INTEGER PRIMARY KEY AUTOINCREMENT,username TEXT UNIQUE COLLATE NOCASE NOT NULL,
            email TEXT UNIQUE COLLATE NOCASE NOT NULL,password_hash BLOB NOT NULL,security_question TEXT,security_answer_hash BLOB,role TEXT DEFAULT 'user',created_at TEXT
        )
        """
    )
    cursor.execute("PRAGMA table_info(users)")
    existing_columns = {row["name"]for row in cursor.fetchall()}
    if "role" not in existing_columns:cursor.execute("ALTER TABLE users ADD COLUMN role TEXT DEFAULT 'user'")
    if "created_at" not in existing_columns:cursor.execute("ALTER TABLE users ADD COLUMN created_at TEXT")
    cursor.execute(
        """
        UPDATE users
        SET created_at = ?
        WHERE created_at IS NULL OR created_at = ''
        """,
        (dt.datetime.now(dt.timezone.utc).isoformat(),),)
    if ADMIN_EMAIL:
        cursor.execute(
            """
            UPDATE users
            SET role = 'user'
            WHERE lower(email) != ?
              AND lower(role) = 'admin'
            """,
            (ADMIN_EMAIL,),
        )

        cursor.execute(
            """
            UPDATE users
            SET role = 'admin'
            WHERE lower(email) = ?
            """,
            (ADMIN_EMAIL,),
        )
    conn.commit()
    conn.close()
init_db()
def valid_email(email):
    """
    Validate email format.
    Digits are allowed in email addresses, for example:
    sneha123@gmail.com
    user2026@example.com
    """
    email = email.strip()
    pattern = (r"^[A-Za-z0-9._%+-]+"r"@[A-Za-z0-9.-]+"r"\.[A-Za-z]{2,}$")
    return bool(re.fullmatch(pattern, email))
def password_error(password):
    """
    Return an error message if the password does not meet
    the security requirements. Return None if valid.
    """
    if len(password) < 8:return "Password must contain at least 8 characters."
    if not re.search(r"[A-Z]", password):return "Password must contain at least one uppercase letter."
    if not re.search(r"[a-z]", password):return "Password must contain at least one lowercase letter."
    if not re.search(r"\d", password):return "Password must contain at least one digit."
    if not re.search(r"[^A-Za-z0-9]", password):return "Password must contain at least one special character."
    return None
def hash_password(password):
    """Hash a password securely using bcrypt."""
    return bcrypt.hashpw(password.encode("utf-8"),bcrypt.gensalt(),)
def verify_password(password, stored_hash):
    """Verify a password against its stored bcrypt hash."""
    if not password or not stored_hash:return False
    try:
        if isinstance(stored_hash, str):stored_hash = stored_hash.encode("utf-8")
        return bcrypt.checkpw(password.encode("utf-8"),stored_hash,)
    except (ValueError, TypeError):return False

def hash_security_answer(answer):
    """Normalize and hash the security-question answer."""
    normalized_answer = answer.strip().lower()
    return bcrypt.hashpw(normalized_answer.encode("utf-8"),bcrypt.gensalt(),)
def verify_security_answer(answer, stored_hash):
    """Verify the user's security-question answer."""
    if not answer or not stored_hash:return False
    try:
        if isinstance(stored_hash, str):stored_hash = stored_hash.encode("utf-8")
        normalized_answer = answer.strip().lower()
        return bcrypt.checkpw(normalized_answer.encode("utf-8"),stored_hash,)
    except (ValueError, TypeError):return False
def make_jwt(email):
    """Create a JWT token valid for two hours."""
    if not JWT_SECRET:
        raise RuntimeError("JWT_SECRET is not configured. ""Set the JWT_SECRET environment variable before login.")
    payload = {
        "email": email.strip().lower(),
        "exp": (dt.datetime.now(dt.timezone.utc)+ dt.timedelta(hours=2)),"iat": dt.datetime.now(dt.timezone.utc),}
    return jwt.encode(payload,JWT_SECRET,algorithm="HS256",)
def verify_jwt(token):
    """Validate a JWT token and return its payload."""
    if not token or not JWT_SECRET:return None
    try:
        return jwt.decode(token,JWT_SECRET,algorithms=["HS256"],)
    except jwt.PyJWTError:return None

def send_otp_email(recipient_email, otp):
    """
    Send a password-reset OTP through Gmail SMTP.
    EMAIL_ADDRESS and EMAIL_PASSWORD must be configured
    as environment variables.
    """
    if not EMAIL_ADDRESS or not EMAIL_PASSWORD:
        return (False,"Email configuration is missing. ""Please configure EMAIL_ADDRESS and EMAIL_PASSWORD.",)
    try:
        message = EmailMessage()
        message["Subject"] = "FreightQuote AI - Password Reset OTP"
        message["From"] = EMAIL_ADDRESS
        message["To"] = recipient_email
        message.set_content(
            f"""
Hello,
Your FreightQuote AI password-reset OTP is:
{otp}
This OTP expires in 5 minutes.
If you did not request a password reset, please ignore this email.
Regards,
FreightQuote AI Team
"""
        )
        with smtplib.SMTP("smtp.gmail.com",587,timeout=20,) as server:
            server.starttls()
            server.login(EMAIL_ADDRESS,EMAIL_PASSWORD,)
            server.send_message(message)
        return True, "OTP sent successfully."
    except Exception:
        return (False,"Unable to send the OTP email. ""Please check your email configuration and try again.",)
SESSION_DEFAULTS = {
    "page": "Login","token": None,"reset_email": "","reset_mode": "","otp_code": None,"otp_expires": 0,
    "otp_verified": False,"sq_email": "","sq_question": "","admin_logged_in": False,}
for key, default_value in SESSION_DEFAULTS.items():
    if key not in st.session_state:st.session_state[key] = default_value
def navigate(page):
    """Change the current page and rerun the app."""
    st.session_state.page = page
    st.rerun()
def clear_reset():
    """Clear password-reset-related session values."""
    st.session_state.reset_email = ""
    st.session_state.reset_mode = ""
    st.session_state.otp_code = None
    st.session_state.otp_expires = 0
    st.session_state.otp_verified = False
    st.session_state.sq_email = ""
    st.session_state.sq_question = ""

def header(title,subtitle="AI-powered Freight Quotation Platform",):
    """Display the common page heading."""
    st.markdown(
        f"""
        <div style="margin-bottom: 24px;">
            <div class="brand-name">FreightQuote AI</div>
            <div class="brand-tagline">
                {subtitle}
            </div>
            <h1 style="font-size: 2rem; margin-bottom: 6px;">
                {title}
            </h1>
        </div>
        """,
        unsafe_allow_html=True,
    )

def login_user(email):
    """Create a JWT token and navigate to the dashboard."""
    email = email.strip().lower()
    token = make_jwt(email)
    st.session_state.token = token
    st.session_state.admin_logged_in = False
    st.session_state.page = "Dashboard"
    st.rerun()

def get_user_by_email(email):
    """Fetch a user by email address."""
    conn = get_connection()
    user = conn.execute(
        """
        SELECT *
        FROM users
        WHERE lower(email) = ?
        """,
        (email.strip().lower(),),).fetchone()
    conn.close()
    return user
def get_user_by_username_or_email(value):
    """Fetch a user by username or email."""
    value = value.strip().lower()
    conn = get_connection()
    user = conn.execute(
        """
        SELECT *
        FROM users
        WHERE lower(email) = ?
           OR lower(username) = ?
        """,
        (value, value),
    ).fetchone()
    conn.close()
    return user
def render_auth_banner():
    """Display the FreightQuote AI welcome section."""
    st.markdown(
        """
        <div class="custom-card"
             style="
                background: linear-gradient(120deg,#E5F2FF 0%,#FFFFFF 65%, #FFF8D9 100%);
                padding: 28px;
             ">
            <div style="font-size: 13px;font-weight: 700;color: #3478F6;letter-spacing: 1.5px;margin-bottom: 12px;
            ">
                SMARTER FREIGHT MANAGEMENT
            </div>
            <h2 style="color: #203B69;font-size: 27px;margin-bottom: 10px;
            ">
                Freight decisions, made simpler.
            </h2>
            <p style="color: #70809A;font-size: 15px;line-height: 1.8;margin-bottom: 0;
            ">
                Access your account to manage freight quotations
                and explore your FreightQuote AI workspace.
            </p>

        </div>
        """,
        unsafe_allow_html=True,
    )
def render_login_page():
    st.markdown(
        """
        <style>
        .main .block-container {max-width: 1100px;padding-top: 0rem;padding-bottom: 3rem;}
        .fq-header {text-align: center;padding: 28px 15px;margin-bottom: 30px;
            background: linear-gradient(120deg, #E8F4FF, #FFFFFF, #FFF8DE);border: 1px solid #D8E8F8;border-radius: 18px;}
        .fq-brand {font-size: 38px;font-weight: 800;color: #15366D;margin-bottom: 8px;}
        .fq-brand span {color: #2782F6;}
        .fq-tagline {color: #6481A9;font-size: 16px;}
        .welcome-card {background: linear-gradient(145deg, #E8F4FF, #FFFFFF, #FFF8DE);border: 1px solid #D8E8F8;border-radius: 18px;padding: 25px;margin-bottom: 20px;}
        .welcome-title {color: #15366D;font-size: 25px;font-weight: 800;margin: 12px 0;}
        .welcome-description {color: #6380A5;font-size: 14px;line-height: 1.8;}
        [data-testid="stTextInput"] input {border-radius: 9px;min-height: 42px;}
        div.stButton > button {border-radius: 9px;min-height: 44px;font-weight: 600;}
        div.stButton > button[kind="primary"] {background-color: #FFD21F;color: #15366D;border: 1px solid #FFD21F;}
        div.stButton > button[kind="primary"]:hover {background-color: #F4C400;border-color: #F4C400;color: #15366D;}
        </style>
        """,
        unsafe_allow_html=True,
    )
    st.markdown(
    """
    <div style="text-align: center;padding: 0;margin-top:-25px;margin-biottom:40px;background: transparent;border: 1px solid #D6E7F8;border:none;box-shadow:none;
    ">
        <h1 style="color: #15366D;font-size: 38px;font-weight: 800;margin: 0;padding:0
        ">
             FreightQuote <span style="color:#2782F6;">AI</span>
        </h1>
    </div>
    """,
    unsafe_allow_html=True,
)
    left_space, login_column, right_space = st.columns([1, 1.7, 1],gap="large",)

    with login_column:
        # Login form
        st.markdown(
            """
            <h2 style="color:#15366D;text-align:center;margin-bottom:5px;
            ">
                Welcome Back!
            </h2>

            <p style="color:#6380A5;text-align:center;margin-bottom:22px;
            ">
                Sign in to your account
            </p>
            """,
            unsafe_allow_html=True,
        )
        with st.container(border=True):
            st.markdown("###  Sign In")
            with st.form("login_form"):
                login_identity = st.text_input("Username or Email",placeholder="Enter your username or email",)
                login_password = st.text_input("Password",type="password",placeholder="Enter your password",)
                remember_me = st.checkbox("Remember me")
                submitted = st.form_submit_button("Sign In  →",type="primary",use_container_width=True,)
            if submitted:
                if not login_identity.strip() or not login_password:
                    st.error("Please enter your username/email and password.")
                else:
                    user = get_user_by_username_or_email(login_identity)
                    if user and verify_password(login_password,user["password_hash"],):
                        try:
                            login_user(user["email"])
                        except RuntimeError as exc:st.error(str(exc))
                    else:
                        st.error("Invalid username/email or password.")
            st.markdown(
                """
                <p style="color:#6380A5;text-align:center;margin-top:15px;
                ">
                    Don't have an account?
                </p>
                """,
                unsafe_allow_html=True,
            )
            if st.button("＋ Create Account",use_container_width=True,):navigate("Signup")
            if st.button("Forgot Password?",use_container_width=True,):
              clear_reset()
              navigate("Forgot")
            st.markdown(
                """
                <div style="border-top:1px solid #DCE8F5;padding-top:16px;margin-top:18px;color:#6380A5;font-size:13px;
                ">
                      <b>Secure Access</b><br>
                    Your account is protected with password hashing
                    and JWT authentication.
                </div>
                """,
                unsafe_allow_html=True,
            )

def render_signup_page():

    st.markdown(
    """
    <style>
    .main .block-container {max-width: 1100px;padding-top: 1rem;}
    </style>
    <div style="text-align: center;padding: 0;
    ">
    <h1 style="font-size: 38px;font-weight: 800;color: #203B69;margin-bottom: 10px;
    ">
        Create Your Account
    </h1>

    <p style="color: #70809A;font-size: 16px;
    ">
        Join FreightQuote AI and set up your workspace.
    </p>
    </div>
    """,
    unsafe_allow_html=True,
)

    left_space, signup_col, right_space = st.columns([1, 1.4, 1])
    with signup_col:
        st.markdown("### Sign Up")
        st.caption("Fill in the details below to create an account.")
        with st.form("signup_form"):
            username = st.text_input("Username",placeholder="Choose a username",)
            email = st.text_input("Email Address",placeholder="example123@gmail.com",)
            password = st.text_input("Password",type="password",placeholder="Create a strong password",)
            confirm_password = st.text_input("Confirm Password",type="password",placeholder="Re-enter your password", )
            st.markdown("#### Account Recovery")
            security_question = st.selectbox("Security Question",SECURITY_QUESTIONS,)
            security_answer = st.text_input("Security Answer", placeholder="Enter your answer",)
            create_account = st.form_submit_button("Create Account",type="primary",use_container_width=True,)
        if create_account:
            username = username.strip()
            email = email.strip().lower()
            if not all([username,email,password,confirm_password,security_answer.strip(),]):
                st.error("Please fill in all the required fields.")
            elif not valid_email(email):st.error("Please enter a valid email address.")
            elif len(username) < 3:st.error("Username must contain at least 3 characters.")
            elif len(username) > 50:st.error("Username cannot exceed 50 characters.")
            elif any(char.isspace() for char in username):st.error("Username cannot contain spaces.")
            elif not re.fullmatch(r"[A-Za-z0-9_.-]+",username,):
                st.error("Username may contain letters, digits, ""underscores, dots and hyphens only.")
            elif password_error(password):st.error(password_error(password))
            elif password != confirm_password:st.error("Passwords do not match.")
            else:
                conn = get_connection()
                try:
                    password_hash = hash_password(password)
                    answer_hash = hash_security_answer( security_answer)
                    conn.execute(
                        """
                        INSERT INTO users (username,email,password_hash,security_question,security_answer_hash,role,created_at)
                        VALUES (?, ?, ?, ?, ?, ?, ?)
                        """,
                        (username,email,password_hash,security_question,answer_hash,"user",dt.datetime.now(dt.timezone.utc).isoformat(),),)
                    conn.commit()
                    st.success("Account created successfully! ""Please sign in.")
                    st.session_state.page = "Login"
                    st.rerun()
                except sqlite3.IntegrityError:
                    st.error("This username or email is already registered.")
                except sqlite3.Error:
                    st.error("Unable to create your account. ""Please try again.")
                finally:
                    conn.close()
        st.markdown("---")
        st.write("Already have an account?")
        if st.button(
            "← Back to Sign In",
            use_container_width=True,
        ):
            navigate("Login")
        st.markdown("</div>", unsafe_allow_html=True)

def render_forgot_page():
    header("Reset your password","Choose a recovery method to regain access to your account.",)
    st.markdown("### Account Recovery")
    reset_method = st.radio("Choose a recovery method",["Security Question", "Email OTP",], horizontal=True, key="reset_method_choice",)
    reset_email = st.text_input("Registered Email Address", value=st.session_state.reset_email,placeholder="Enter your registered email",key="forgot_email_input",)
    if reset_method == "Security Question":
        if st.button("Continue with Security Question",type="primary",use_container_width=True,):
            email = reset_email.strip().lower()
            if not valid_email(email):
                st.error("Please enter a valid registered email address.")
            else:
                user = get_user_by_email(email)
                if not user:st.error( "No account was found with this email.")
                elif not user["security_question"]:
                    st.error("No security question is configured ""for this account.")
                else:
                    st.session_state.sq_email = email
                    st.session_state.sq_question = ( user["security_question"])
                    st.session_state.reset_email = email
                    st.session_state.reset_mode = ("Security Question")
                    navigate("ForgotSecurity")
    else:
        if st.button("Send Email OTP",type="primary", use_container_width=True,):
            email = reset_email.strip().lower()
            if not valid_email(email):st.error("Please enter a valid registered email address.")
            else:
                user = get_user_by_email(email)
                if not user:
                    st.error("No account was found with this email.")
                else:
                    otp = f"{secrets.randbelow(1000000):06d}"
                    success, message = send_otp_email(email,otp,)
                    if success:
                        st.session_state.reset_email = email
                        st.session_state.reset_mode = "Email OTP"
                        st.session_state.otp_code = otp
                        st.session_state.otp_expires = (dt.datetime.now(dt.timezone.utc).timestamp()+ OTP_EXPIRY_SECONDS)
                        st.session_state.otp_verified = False
                        navigate("ForgotOTP")
                    else:
                        st.error(message)

    st.markdown("---")
    if st.button(
        "← Back to Sign In",
        use_container_width=True,
    ):
        clear_reset()
        navigate("Login")
    st.markdown("</div>", unsafe_allow_html=True)

def render_security_reset_page():
    header("Verify your identity","Answer your security question to reset your password.",)
    email = st.session_state.sq_email
    question = st.session_state.sq_question
    if not email or not question:
        st.warning("Your recovery session has expired. ""Please start again.")
        if st.button("Return to Sign In"):clear_reset()
        navigate("Login")
        return
    st.write("Registered email:")
    st.info(email)
    st.markdown("#### Security Question")
    st.write(question)
    with st.form("security_reset_form"):
        answer = st.text_input("Your Answer", placeholder="Enter your security answer",)
        new_password = st.text_input("New Password",type="password", placeholder="Enter your new password",)
        confirm_password = st.text_input("Confirm New Password",type="password", placeholder="Re-enter your new password",)
        reset_button = st.form_submit_button("Reset Password",type="primary",use_container_width=True, )
    if reset_button:
        user = get_user_by_email(email)
        if not user:st.error("Account not found.")
        elif not answer.strip():st.error("Please enter your security answer.")
        elif not verify_security_answer(answer,user["security_answer_hash"],):st.error("Incorrect security answer.")
        elif password_error(new_password):st.error(password_error(new_password))
        elif new_password != confirm_password:st.error("Passwords do not match.")
        else:
            conn = get_connection()
            try:
                conn.execute(
                    """
                    UPDATE users
                    SET password_hash = ?
                    WHERE lower(email) = ?
                    """,
                    (hash_password(new_password),email,),)
                conn.commit()
                clear_reset()
                st.success("Password reset successfully. ""Please sign in with your new password.")
                st.session_state.page = "Login"
                st.rerun()
            finally:
                conn.close()
    st.markdown("---")
    if st.button("Cancel",use_container_width=True, ):
      clear_reset()
      navigate("Login")
    st.markdown("</div>", unsafe_allow_html=True)

def render_otp_reset_page():
    header("Verify email OTP","Enter the verification code sent to your registered email.",)
    email = st.session_state.reset_email
    if not email or st.session_state.otp_code is None:
        st.warning("No active OTP request was found. ""Please request a new OTP.")
        if st.button("Return to Password Recovery"):
          clear_reset()
          navigate("Forgot")
        return
    st.write("OTP sent to:")
    st.info(email)
    if not st.session_state.otp_verified:
        with st.form("otp_verification_form"):
            entered_otp = st.text_input("6-digit OTP",max_chars=6,placeholder="Enter the OTP",)
            verify_button = st.form_submit_button("Verify OTP",type="primary", use_container_width=True,)
        if verify_button:
            current_timestamp = (dt.datetime.now(dt.timezone.utc ).timestamp())
            if current_timestamp > st.session_state.otp_expires:
              st.error("OTP expired. Please request a new one.")
              st.session_state.otp_code = None
              st.session_state.otp_verified = False
            elif not re.fullmatch(r"\d{6}", entered_otp.strip(),):
              st.error("Please enter a valid 6-digit OTP.")
            elif not secrets.compare_digest(entered_otp.strip(), st.session_state.otp_code,):
              st.error("Incorrect OTP. Please try again.")
            else:
                st.session_state.otp_verified = True
                st.success("Email verified successfully. ""You can now reset your password.")
                st.rerun()

    if st.session_state.otp_verified:
        with st.form("otp_new_password_form"):
            new_password = st.text_input("New Password",type="password",placeholder="Enter your new password",)
            confirm_password = st.text_input( "Confirm New Password",type="password",placeholder="Re-enter your new password",)
            reset_button = st.form_submit_button("Update Password",type="primary",use_container_width=True,)
        if reset_button:
            current_timestamp = (dt.datetime.now( dt.timezone.utc).timestamp())
            if current_timestamp > st.session_state.otp_expires:
                st.error("OTP verification has expired. " "Please request a new OTP.")
                clear_reset()
                st.session_state.page = "Forgot"
                st.rerun()
            elif password_error(new_password):st.error(password_error(new_password))
            elif new_password != confirm_password:
                st.error("Passwords do not match.")
            else:
                conn = get_connection()
                try:
                    conn.execute(
                        """
                        UPDATE users
                        SET password_hash = ?
                        WHERE lower(email) = ?
                        """,
                        (hash_password(new_password),email,), )
                    conn.commit()
                    clear_reset()
                    st.success("Password updated successfully. ""Please sign in.")
                    st.session_state.page = "Login"
                    st.rerun()
                finally:
                    conn.close()
    st.markdown("---")
    if st.button("Cancel",use_container_width=True,):
      clear_reset()
      navigate("Login")
    st.markdown("</div>", unsafe_allow_html=True)
def render_metric_card(label, value, description):
    st.markdown(
        f"""
        <div class="metric-card">
            <div class="metric-label">
                {label}
            </div>

            <div class="metric-value">
                {value}
            </div>

            <div style="color: #70809A;font-size: 12px;margin-top: 6px;
            ">
                {description}
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )

def render_user_dashboard(user):

    header("Dashboard","Your FreightQuote AI workspace at a glance.",)
    username = user["username"]
    email = user["email"]
    st.markdown(
        f"""
        <div style="background: linear-gradient(110deg,#E5F2FF,#FFFFFF,#FFF9E3);border: 1px solid #D6E7F8;border-radius: 16px;padding: 24px; margin-bottom: 20px;
        ">
            <h2 style="color: #15366D;margin-top: 0;margin-bottom: 10px;
            ">
                Welcome, {username}!
            </h2>

          <p style="color: #6380A5;font-size: 15px;margin-bottom: 0;
          ">
              You're signed in to FreightQuote AI.
               Here's an overview of your workspace.
          </p>
        </div>
        """,
        unsafe_allow_html=True,
    )

    col1, col2, col3, col4 = st.columns(4)
    with col1:st.metric(label="Workspace",value="Active",help="Your account is ready",)
    with col2:st.metric(label="Account",value="Verified",help="Authenticated session",)
    with col3:st.metric(label="Platform",value="AI",help="Freight quotation workspace",)
    with col4:st.metric(label="Access",value="User",help="Standard account access",)
    st.markdown("## Workspace Overview")
    with st.container(border=True):
      st.markdown("###  Get started")
      st.write("Your dashboard is ready. You can use this ""workspace to organize your freight-related ""activities as the platform's features become ""available." )
    st.markdown("## Account Details")
    col1, col2 = st.columns(2)
    with col1:
        with st.container(border=True):
          st.caption("USERNAME")
          st.markdown(f"### {username}")
    with col2:
      with st.container(border=True):
        st.caption("REGISTERED EMAIL")
        st.markdown(f"**{email}**")

def render_admin_dashboard(user):
    header("Admin Dashboard","Manage and monitor registered platform accounts.",)
    st.markdown(
        """
        <div class="custom-card"
             style="
                background: linear-gradient(110deg,#E5F2FF,#FFFFFF,#FFF9E3 );
             ">
            <h2 style="margin-top: 0;">
                Welcome, Administrator
            </h2>
            <p class="subtitle">
                You have administrative access to the
                FreightQuote AI user overview.
            </p>

        </div>
        """,
        unsafe_allow_html=True,
    )
    conn = get_connection()
    try:
        total_users = conn.execute("SELECT COUNT(*) FROM users").fetchone()[0]
        total_admins = conn.execute(
            """
            SELECT COUNT(*)
            FROM users
            WHERE lower(role) = 'admin'
            """
        ).fetchone()[0]
        users = conn.execute(
            """
            SELECT id,username,email,role,created_at FROM users ORDER BY id DESC
            """
        ).fetchall()
    finally:
        conn.close()
    col1, col2, col3 = st.columns(3)
    with col1:render_metric_card("Total Accounts",total_users,"All registered accounts",)
    with col2:render_metric_card("Administrators",total_admins,"Accounts with admin access",)
    with col3:render_metric_card("Regular Users",total_users - total_admins,"Standard platform accounts",)
    st.markdown("### Registered Users")
    if users:
        st.dataframe([{"ID": row["id"],"Username": row["username"],"Email": row["email"],"Role": row["role"],"Created At": row["created_at"],}for row in users],use_container_width=True,hide_index=True,)
    else:
        st.info("No registered users were found.")

def render_dashboard():
    token = st.session_state.token
    payload = verify_jwt(token)
    if not payload:
        st.session_state.token = None
        st.session_state.admin_logged_in = False
        st.warning( "Your session has expired. Please sign in again.")
        st.session_state.page = "Login"
        st.rerun()
    email = payload.get("email", "").strip().lower()
    user = get_user_by_email(email)
    if not user:
        st.session_state.token = None
        st.session_state.admin_logged_in = False
        st.error("Your account could not be found. Please sign in again.")
        st.session_state.page = "Login"
        st.rerun()
    is_admin = (str(user["role"]).strip().lower() == "admin")
    st.sidebar.markdown(
        """
        <div style="padding: 14px 8px 20px 8px;
        ">
        <div style="ont-size: 21px;font-weight: 800;color: #2458B8;
        ">
             FreightQuote AI
        </div>
        <div style="color: #70809A;font-size: 12px;margin-top: 5px;
        ">
            Your freight workspace
        </div>
        </div>
        """,
        unsafe_allow_html=True,
    )
    st.sidebar.markdown("---")
    with st.sidebar:
        selected = option_menu(menu_title="Navigation",options=["Dashboard","Logout",], icons=["speedometer2","box-arrow-right",],default_index=0,
            styles={
                "container": {"padding": "5px","background-color": "transparent",},
                "icon": {"color": "#3478F6","font-size": "17px",},
                "nav-link": {"font-size": "14px","text-align": "left","margin": "5px 0","border-radius": "10px","color": "#263858",},
                "nav-link-selected": {"background-color": "#3478F6","color": "#FFFFFF","font-weight": "600",},},)
        st.markdown("---")
        st.caption(f"Signed in as: {user['username']}")
        st.caption("Role: Administrator" if is_admin else "Role: User")
    if selected == "Logout":
        st.session_state.token = None
        st.session_state.admin_logged_in = False
        clear_reset()
        st.session_state.page = "Login"
        st.rerun()
    if is_admin:
        render_admin_dashboard(user)
    else:
        render_user_dashboard(user)
def main():
    if st.session_state.token:
        render_dashboard()
        return
    current_page = st.session_state.page
    if current_page == "Login":render_login_page()
    elif current_page == "Signup":render_signup_page()
    elif current_page == "Forgot":render_forgot_page()
    elif current_page == "ForgotSecurity":render_security_reset_page()
    elif current_page == "ForgotOTP":render_otp_reset_page()
    else:
        st.session_state.page = "Login"
        render_login_page()

if __name__ == "__main__":
    main()


Overwriting app.py


In [10]:
from google.colab import userdata
import os
required = ["JWT_SECRET", "NGROK_AUTHTOKEN", "EMAIL_ADDRESS", "EMAIL_PASSWORD"]
optional = ["ADMIN_EMAIL"]
missing = []
for key in required:
    try:
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        missing.append(key)
    else:
        os.environ[key] = value

for key in optional:
    try:
        value = userdata.get(key)
        if value:
            os.environ[key] = value
    except Exception:
        pass

if missing:
    raise RuntimeError("Add these required keys in Colab Secrets and enable notebook access: " + ", ".join(missing))

from pyngrok import ngrok
import subprocess, time, sys
ngrok.kill()
subprocess.run("pkill -f streamlit", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])
process = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
time.sleep(4)
tunnel = ngrok.connect(8501, "http")
print("FreightQuote AI URL:", tunnel.public_url)
print("Keep this cell running while using the app. Stop the cell to end the session.")
try:
    while process.poll() is None:
        time.sleep(1)
except KeyboardInterrupt:
    pass
finally:
    process.terminate()
    ngrok.kill()

FreightQuote AI URL: https://neuter-slot-poncho.ngrok-free.dev
Keep this cell running while using the app. Stop the cell to end the session.
